**Workshop: Análise de Dados e Ajustes Experimentais em Python**

# 1 — Estatística básica e incertezas

Neste notebook revemos quantidades estatísticas que vão aparecer durante todo o workshop: média, variância, desvio padrão, covariância, correlação, e como propagar incertezas.

Os dados que vamos usar são sintéticos, mas correspondem ao que se obtém de um **cintilador acoplado a um analisador multicanal (MCA)** — um espectro de contagens por canal, como vão ver em LFEA I e II :) 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import chi2

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

## 1.1 — Estatísticas de uma amostra

Considerando uma variável aleatória $X$ com valor esperado $\mu$ e variância $\sigma^2$, a média amostral $\bar{X}$ é um estimador de $\mu$, e a variância amostral $S^2$ é um estimador de $\sigma^2$.

Dadas $N$ medições $x_i$:

$$
\bar{X} = \frac{1}{N}\sum_i X_i, \qquad
S^2 = \frac{1}{N-1}\sum_i (X_i - \bar{X})^2, \qquad
S = \sqrt{S^2}
$$

Neste caso, são ambos estimadores **centrados**.

A **incerteza da média** (erro padrão) é $\sigma_{\bar{X}} = S/\sqrt{N}$.

Considerando uma distribuição de Poisson:
$$
P(X=k) = \frac{\lambda^k e^{-\lambda}}{k!} 
$$

$X\sim \text{Poisson}(\lambda)$ é uma variável aleatória que indica o numero de ocorrências de um acontecimento por unidade de tempo ou espaço. 
Por exemplo, o número de partículas detectadas por um fotomultiplicador por canal em um MCA segue uma distribuição de Poisson, onde $\lambda$ é a média de contagens por canal.

A média e a variância de uma distribuição de Poisson são ambas iguais a $\lambda$:
$$
\mathbb{E}[X] = \lambda, \qquad \text{Var}(X) = \lambda
$$

Quais será a variância amostral da média de $N$ medições independentes de uma variável aleatória $X$ que segue uma distribuição de Poisson?
<!-- $$
\bar{X} = \frac{1}{N}\sum_i X_i \approx \lambda,
$$
Recordando que $\text{Var}(\alpha X) = \alpha^2 \text{Var}(X)$ e $\text{Var}(X+Y) = \text{Var}(X) + \text{Var}(Y)$ para variáveis independentes, temos:
$$
\text{Var}(\bar{X}) = \text{Var}\left(\frac{1}{N}\sum_i X_i\right) = \frac{1}{N^2} \sum_i \text{Var}(X_i) = \frac{1}{N^2} \sum_i \lambda = \frac{\lambda}{N} \approx \frac{\bar{X}}{N}
$$
Portanto, a variância amostral $S^2$ é um estimador de $\lambda$, e o erro padrão da média é $\sigma_{\bar{X}} = \sqrt{\lambda/N}$. -->

In [ ]:
# TODO: Simula 200 medidas de um evento que segue uma distribuição de Poisson com média 5.0 contagens.
rng = np.random.default_rng(0)
# x = rng.poisson(...)

# media = ...
# var = ...
# std = ...
# sem = ...

print(f"Média        = {media:.2f}")
print(f"Variância    = {var:.2f}")
print(f"Desvio padrão = {std:.2f}")
print(f"Erro da média = {sem:.2f}")
print(f"\nResultado: ({media:.2f} ± {sem:.2f}) contagens")

Qual é a diferença entre a variância amostral $S^2$ e a variância da média $\text{Var}(\bar{X})$?
A variância amostral $S^2$ é um estimador da variância populacional $\sigma^2$, enquanto a variância da média $\text{Var}(\bar{X})$ é a variância do estimador da média, que é igual a $\sigma^2/N$. Portanto, $S^2$ estima a variabilidade dos dados individuais, enquanto $\text{Var}(\bar{X})$ estima a variabilidade da média amostral em torno do valor verdadeiro $\mu$. Podemos mostrar facilmente com um exemplo numérico utilizando técnicas de simulação de Monte Carlo.

In [ ]:
lambda_real = 100
N = 9

rng = np.random.default_rng(0)
experiencias = rng.poisson(lambda_real, size=(10000, N))
medias = experiencias.mean(axis=1)

sigma_real = np.sqrt(lambda_real)  # ← raiz quadrada!

print(f"std teórico (= √λ):              {sigma_real:.3f}")
print(f"std amostral (1 experiência):    {experiencias[0].std(ddof=1):.3f}")
print(f"std(x̄) medido (10000 médias):    {medias.std():.3f}")
print(f"std(x̄) previsto (= √(λ/N)):       {sigma_real / np.sqrt(N):.3f}")

> ⚠️ **Atenção ao `ddof`**: `np.std(x)` usa por defeito `ddof=0` (divide por $N$). Para uma estimativa não-enviesada da variância de uma amostra usa sempre `ddof=1`.

Como visualizar os dados? Podemos usar um histograma, que é uma representação gráfica da distribuição de frequências de um conjunto de dados. O eixo horizontal representa os valores dos dados, enquanto o eixo vertical representa a frequência ou contagem de cada valor ou intervalo de valores (bin).

In [ ]:
# Histograma com a media e ±1σ assinalados
fig, ax = plt.subplots()
ax.hist(x, bins=20, edgecolor="k", alpha=0.7)
ax.axvline(media, color="r", lw=2, label=f"média = {media:.1f}")
ax.axvspan(media - std, media + std, color="r", alpha=0.15, label=r"$\pm 1 \sigma$")
ax.set_xlabel("contagens")
ax.set_ylabel("frequência")
ax.legend()
plt.show()


## 1.2 — Barras de erro: estatística de Poisson

Para contagens (eventos discretos independentes) a distribuição é **Poisson**, em que $\sigma = \sqrt{\lambda}$. 
Neste caso, temos $n$ contagens por canal, e so temos uma medição, então a incerteza é $\sigma = \sqrt{n}$.

In [ ]:
espectro = np.loadtxt("../data/espectro_mca.csv", delimiter=",", skiprows=1)
canal, contagens = espectro[:, 0], espectro[:, 1]
sigma = np.sqrt(np.maximum(contagens, 1))  # evita σ=0 em canais vazios

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(canal, contagens, yerr=sigma, fmt=".", ms=2, lw=0.5, ecolor="gray", label=r"dados $\pm~\sqrt{N}$")
ax.set_xlabel("Canal")
ax.set_ylabel("Contagens")
ax.set_title(r"Espectro do MCA ($^{137}$Cs + $^{60}$Co)")
ax.legend()
plt.show()

## 1.3 — Covariância e correlação

Para duas variáveis $x, y$:

$$
\text{cov}(x,y) = \frac{1}{N-1}\sum_i (x_i-\bar x)(y_i-\bar y),\qquad
\rho_{xy} = \frac{\text{cov}(x,y)}{s_x s_y} \in [-1,1]
$$

Em `numpy` usamos `np.cov` e `np.corrcoef` (devolvem **matrizes**).

No caso da matriz de covariância, os elementos diagonais são as variâncias de cada variável, e os elementos fora da diagonal são as covariâncias entre as variáveis:

$$
C = \begin{bmatrix}
\text{Var}(x) & \text{cov}(x,y) \\
\text{cov}(y,x) & \text{Var}(y)
\end{bmatrix}
$$

No caso da matriz de correlação, os elementos diagonais são 1 (correlação perfeita de cada variável consigo mesma), e os elementos fora da diagonal são as correlações entre as variáveis:
$$R = \begin{bmatrix}
1 & \rho_{xy} \\
\rho_{yx} & 1
\end{bmatrix}
$$

In [ ]:
# Exemplo: pequena calibracao (canal vs energia conhecida)
calib = np.genfromtxt("../data/calibracao.csv", delimiter=",", skip_header=1, usecols=(1, 3))
canal_cal, energia_cal = calib[:, 0], calib[:, 1]

C = np.cov(canal_cal, energia_cal)
R = np.corrcoef(canal_cal, energia_cal)
print("Matriz de covariância:\n", C)
print("\nMatriz de correlação:\n", R)
print(f"\nrho(canal, energia) = {R[0, 1]:.4f}")

Estes valores fazem sentido?

## 1.3.1 — Da covariância à propagação de incertezas

A matriz de covariância **não é uma curiosidade matemática** — é exactamente o objecto que precisas para responder à pergunta:

> *Se $f$ depende de várias quantidades medidas com incerteza, qual é a incerteza em $f$?*

Esta é a chamada **fórmula de propagação de incertezas**. Vamos derivá-la a partir do zero.

### Setup

Tens uma função de várias variáveis medidas:

$$f = f(x_1, x_2, \dots, x_n)$$

Cada $x_i$ é uma variável aleatória com média $\mu_i = \langle x_i\rangle$ e variância $\sigma_i^2 = \text{Var}(x_i)$. Como $f$ depende dos $x_i$ (que flutuam), $f$ **também é uma variável aleatória** — queremos saber a sua variância $\sigma_f^2$.

### Passo 1 — Expansão de Taylor

Se as flutuações $\delta x_i \equiv x_i - \mu_i$ forem **pequenas**, expandimos $f$ até primeira ordem à volta dos valores médios:

$$
f(x_1, \dots, x_n) \;\approx\; f(\mu_1, \dots, \mu_n) \;+\; \sum_{i=1}^n \left.\frac{\partial f}{\partial x_i}\right|_{\vec\mu} \delta x_i \;+\; \mathcal{O}(\delta x^2)
$$

Para abreviar, escreve-se $f_i \equiv \partial f / \partial x_i$ avaliado em $\vec\mu$ (são **números**, não variáveis aleatórias):

$$
f \;\approx\; f(\vec\mu) \;+\; \sum_i f_i\,\delta x_i
$$

### Passo 2 — Média de $f$

Tirando valor esperado e usando $\langle \delta x_i\rangle = 0$ (por definição de média):

$$
\langle f\rangle \;\approx\; f(\vec\mu) \;+\; \sum_i f_i \langle\delta x_i\rangle \;=\; f(\vec\mu)
$$

→ Em primeira ordem, **a média da função é a função das médias**.

### Passo 3 — Variância de $f$

Por definição,

$$
\sigma_f^2 \;=\; \langle (f - \langle f\rangle)^2\rangle \;\approx\; \left\langle\left(\sum_i f_i\,\delta x_i\right)^2\right\rangle
$$

Expandindo o quadrado:

$$
\left(\sum_i f_i\,\delta x_i\right)^2 \;=\; \sum_i f_i^2 (\delta x_i)^2 \;+\; \sum_{i\ne j} f_i f_j\,\delta x_i\,\delta x_j
$$

Tirando o valor esperado termo a termo, e reconhecendo que:

- $\langle (\delta x_i)^2\rangle = \text{Var}(x_i) = \sigma_i^2$
- $\langle \delta x_i\,\delta x_j\rangle = \text{cov}(x_i, x_j) \equiv \sigma_{ij}$

obtemos a **fórmula geral**:


### Fórmula geral (com correlações)

$$
\boxed{\;\sigma_f^2 \;=\; \sum_{i=1}^n \left(\frac{\partial f}{\partial x_i}\right)^2 \sigma_i^2 \;+\; 2\sum_{i<j} \left(\frac{\partial f}{\partial x_i}\right)\left(\frac{\partial f}{\partial x_j}\right)\sigma_{ij}\;}
$$

(O factor 2 aparece porque a soma dupla $\sum_{i\ne j}$ conta cada par duas vezes; $\sum_{i<j}$ conta uma só.)

### Forma matricial

Definindo o vector gradiente $\nabla f = (f_1, \dots, f_n)^T$ e a matriz de covariância $\Sigma$ (a mesma que `np.cov` devolve!), a fórmula compacta-se em:

$$
\sigma_f^2 \;=\; (\nabla f)^T\,\Sigma\,(\nabla f)
$$

→ **É exactamente esta operação que o `curve_fit` faz internamente** para devolver a `pcov` dos parâmetros do ajuste, no notebook 2.

### Caso particular: variáveis independentes

Se todos os $\sigma_{ij} = 0$ para $i \ne j$, os termos cruzados desaparecem:

$$
\sigma_f^2 \;=\; \sum_{i=1}^n \left(\frac{\partial f}{\partial x_i}\right)^2 \sigma_i^2
$$


### Verificação numérica: a fórmula funciona mesmo?

Antes de acreditar cegamente na fórmula, vamos **testá-la numericamente** com um exemplo simples — produto de duas variáveis, $f = xy$.

A fórmula prevê (caso independente):
$$\sigma_f^2 = y^2 \sigma_x^2 + x^2 \sigma_y^2$$

A ideia do **Monte Carlo**: em vez de fazer cálculos analíticos, geramos milhares de amostras de $x$ e $y$ das suas distribuições, calculamos $f$ em cada uma, e medimos directamente $\sigma_f$ a partir do histograma.

In [ ]:
# TODO: Calcula o erro de f = x·y usando a fórmula de propagação de erros e compara com a simulação Monte Carlo.

# Valores centrais e incertezas
x_mu, x_sigma = 10.0, 0.5  # x = 10.0 ± 0.5
y_mu, y_sigma = 4.0, 0.2  # y = 4.0 ± 0.2

# --- Predição analítica ---
f_mu = ...
sigma_f_analitico = ...

# --- Monte Carlo: 100k amostras ---
rng = np.random.default_rng(42)
N = 100000
x_samples = rng.normal(x_mu, x_sigma, N)
y_samples = rng.normal(y_mu, y_sigma, N)
f_samples = x_samples * y_samples

sigma_f_mc = np.std(f_samples, ddof=1)

print(f"f = x·y = {f_mu:.2f}")
print(f"  std_f (fórmula)     = {sigma_f_analitico:.4f}")
print(f"  std_f (Monte Carlo) = {sigma_f_mc:.4f}")
print(f"  diferença         = {100 * (sigma_f_mc - sigma_f_analitico) / sigma_f_analitico:+.2f} %")

# Histograma para visualizar
fig, ax = plt.subplots()
ax.hist(f_samples, bins=80, density=True, alpha=0.7, edgecolor="k")
ax.axvline(f_mu, color="r", lw=2, label=f"média = {f_mu:.2f}")
ax.axvspan(f_mu - sigma_f_mc, f_mu + sigma_f_mc, color="r", alpha=0.2, label=rf"$\pm 1\sigma$ (MC) = {sigma_f_mc:.3f}")
ax.set_xlabel("f = x · y")
ax.set_ylabel("densidade")
ax.legend()


### Exercício 1.
1. Calcula a média de contagens nos canais 100–200 do espectro e a respectiva incerteza (assumindo Poisson).
2. Faz o histograma dessas contagens — é compatível com uma gaussiana?

### Exercício 2.
Aplica a fórmula da secção 1.4 a dados reais. Considera o ajuste linear $y = ax + b$, e os seguintes valores:

- $a = 2.00 \pm 0.01$ 
- $b = 5.0  \pm 1.5$
- $\sigma_{ab} = -0.012$  (fortemente anti-correlacionados!)
- canal medido: $x = 330 \pm 1$

**Pergunta:** qual é o valor de $y$ e a sua incerteza?

In [ ]:
# Valores
a, sa = 2.00, 0.01
b, sb = 5.0, 1.5
cov_ab = -0.012
x, sx = 330, 1.0

# --- TODO: completa as linhas abaixo ---
# y = ...
# var_y_sem_corr = ...
# var_y_com_corr = ...
print(f"y = {y:.2f}")

print(f"σ_y sem covariância = {np.sqrt(var_y_sem_corr):.3f}")
print(f"σ_y com covariância = {np.sqrt(var_y_com_corr):.3f}")
print(f"\nIgnorar a covariância sobrestima a incerteza em {100 * (np.sqrt(var_y_sem_corr) / np.sqrt(var_y_com_corr) - 1):.1f} %")

# 2 — Ajustes Lineares e Polinomiais

## 2.1 — Ajuste linear com `curve_fit`

`scipy.optimize.curve_fit` minimiza

$$\chi^2(\vec{\theta}) = \sum_i \left(\frac{y_i - f(x_i;\vec{\theta})}{\sigma_i}\right)^2$$

e devolve **(1)** os parâmetros óptimos `popt` e **(2)** a matriz de covariância `pcov`. Os erros 1σ dos parâmetros são $\sqrt{\text{diag}(\text{pcov})}$.

> Passa **sempre** `sigma=` e `absolute_sigma=True` se as tuas barras de erro forem absolutas — caso contrário o `scipy` reescala-as para que $\chi^2_\text{red}=1$.

In [ ]:
# Gerar dados sintéticos
rng = np.random.default_rng(seed=42)
x = np.linspace(0, 100, 5)
y = 0.3 * x + 1.0 + rng.normal(loc=0, scale=5.0, size=x.shape)  # adicionar ruído ao modelo verdadeiro para o dataset real
# erro em y
sy = np.full_like(x, 5.0)  # incerteza constante de 5 unidades em y

# Teste de Sanidade
plt.scatter(x, y)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Dados Sintéticos")
plt.show()


In [ ]:
def reta(x, a, b):
    return a * x + b


# TODO: Ajusta y = ax + b aos dados sintéticos, usando as incertezas sc como pesos
popt, pcov = curve_fit(...)
a, b = popt
sa, sb = np.sqrt(np.diag(pcov))
cov_ab = pcov[0, 1]

print(f"a = ({a:.5f} ± {sa:.5f})")
print(f"b = ({b:.3f} ± {sb:.3f})")
print(f"cov(a,b) = {cov_ab:.3e}")
print(f"corr(a,b) = {cov_ab / (sa * sb):+.3f}")
print()


## 2.2 — χ², χ² reduzido e p-value

$$
\chi^2 = \sum_i \left(\frac{y_i - f(x_i)}{\sigma_i}\right)^2,\qquad
\chi^2_\text{red} = \frac{\chi^2}{N - p}
$$

com $N$ pontos e $p$ parâmetros livres. Temos 5 pontos e 2 parâmetros ($a$ e $b$), logo $N - p = 3$ graus de liberdade.

Heurística:

- $\chi^2_\text{red} \approx 1$ → modelo consistente com os dados
- $\gg 1$ → modelo mau ou incertezas subestimadas
- $\ll 1$ → incertezas sobrestimadas (ou *overfitting*)

O p-value (`1 - chi2.cdf(chi2, dof)`) quantifica a probabilidade de obter um $\chi^2$ tão grande ou maior por acaso.

Resíduo - a diferença entre os dados e o modelo ajustado, $r_i = y_i - f(x_i)$. O resíduo é uma medida de quão bem o modelo se encaixa aos dados. Se os resíduos são pequenos e aleatoriamente distribuídos em torno de zero, o modelo é considerado um bom ajuste. Se os resíduos mostram um padrão sistemático, isso pode indicar que o modelo não é adequado para os dados.

$$
r_i = y_i - f(x_i)
$$

In [ ]:
# TODO: Calcula os resíduos no eixo y (canal), e o valor de χ², os graus de liberdade (gl), o χ² reduzido e o p-valor do ajuste.
# residuos = ...
# chi2_val = ...
# dof = ...
# chi2_red = ...
# pvalue = 1 - chi2.cdf(chi2_val, dof)

print(f"χ²        = {chi2_val:.2f}")
print(f"gl (dof)  = {dof}")
print(f"χ²/dof    = {chi2_red:.2f}")
print(f"p-value   = {pvalue:.3f}")


## 2.3 — Como apresentar o resultado? Plot do ajuste + resíduos

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(8, 6), gridspec_kw={"height_ratios": [3, 1]})

x_grid = np.linspace(x.min() - 10, x.max() + 10, 200)
ax1.errorbar(x, y, yerr=sy, fmt="o", capsize=3, label="dados")
ax1.plot(x_grid, reta(x_grid, *popt), "r-", label=f"$y = ({a:.4f})\\,x + ({b:.2f})$")
ax1.set_ylabel("y")
ax1.legend()
ax1.set_title(rf"$\chi^2$/dof = {chi2_red:.2f}, p = {pvalue:.2f}")

ax2.errorbar(x, residuos, yerr=sy, fmt="o", capsize=3)
ax2.axhline(0, color="r", lw=1)
ax2.set_xlabel("x")
ax2.set_ylabel("resíduo")
fig.tight_layout()

# 3 — Ajuste de picos: gaussianas e lorentzianas

Em física experimental, é comum ter que lidar com ajustes de distribuições gaussianas.

Por exemplo, se quisermos extrair informação física dos fotopicos de um espectro de contagens de um MCA, podemos modelar cada pico como uma gaussiana (ou lorentziana, dependendo do processo físico subjacente). A posição do pico nos dá a energia da emissão, a largura do pico está relacionada com a resolução do detector, e a área sob o pico pode ser usada para estimar a taxa de contagem.


**Modelos:**

$$
G(x; A,\mu,\sigma) = A\exp\!\left[-\tfrac{1}{2}\left(\tfrac{x-\mu}{\sigma}\right)^2\right]
\qquad\text{FWHM}_G = 2\sqrt{2\ln 2}\,\sigma \approx 2.355\,\sigma
$$

$$
L(x; A,x_0,\gamma) = A\,\frac{\gamma^2}{(x-x_0)^2 + \gamma^2}
\qquad\text{FWHM}_L = 2\gamma
$$

In [ ]:
espectro = np.loadtxt("../data/espectro_mca.csv", delimiter=",", skiprows=1)
canal, contagens = espectro[:, 0], espectro[:, 1]
sigma_y = np.sqrt(np.maximum(contagens, 1))

plt.errorbar(canal, contagens, yerr=sigma_y, fmt=".", ms=2, lw=0.5, ecolor="gray", label=r"dados $\pm~\sqrt{N}$")
plt.xlabel("Canal")
plt.ylabel("Contagens")
plt.title(r"Espectro do MCA ($^{137}$Cs + $^{60}$Co)")
plt.legend()
plt.show()

Como extrair a posição dos picos?
1) Isolar o pico (definir um intervalo de ajuste)
2) Ajustar a gaussiana/lorentziana
3) Extrair os parâmetros de interesse ($\mu$ ou $x_0$ para energia, $\sigma$ ou $\gamma$ para resolução, $A$ para taxa de contagem)

In [ ]:
# 1) Isolar o pico (definir um intervalo de ajuste)
plt.errorbar(canal, contagens, yerr=sigma_y, fmt=".", ecolor="gray", label=r"dados $\pm~\sqrt{N}$")
plt.xlabel("Canal")
plt.ylabel("Contagens")
plt.title(r"Espectro do MCA ($^{137}$Cs + $^{60}$Co)")
plt.legend()
plt.xlim(290, 370)
plt.show()

In [ ]:
# TODO: Define a função de ajuste (gaussiana + constante) e ajusta-a aos dados do pico, usando as incertezas sigma_y como pesos.
# Calcula os parâmetros do ajuste, suas incertezas, a FWHM, a área do pico, o valor de χ², os graus de liberdade, o χ² reduzido e o p-valor do ajuste.


def gauss_const(x, A, mu, sigma, c0):
    return ...


# Pico do Cs-137 (~ canal 328)
mask = (canal > 280) & (canal < 380)
x, y, sy = canal[mask], contagens[mask], sigma_y[mask]

p0 = [y.max() - np.median(y), x[y.argmax()], 10, np.median(y)]
# popt, pcov = curve_fit(...)
perr = np.sqrt(np.diag(pcov))

# A, mu, sg, c0 = popt
# FWHM = ...
# area = ...

# chi_2 = ...
# dof = ...
# chi_2_red = ...
p_value = 1 - chi2.cdf(chi_2, dof)

print(f"Centróide μ     = ({mu:.2f} ± {perr[1]:.2f}) canal")
print(f"Largura   σ     = ({abs(sg):.2f} ± {perr[2]:.2f}) canal")
print(f"FWHM            = {FWHM:.2f} canal")
print(f"Amplitude A     = ({A:.1f} ± {perr[0]:.1f})")
print(f"Fundo     c0    = ({c0:.1f} ± {perr[3]:.1f})")
print(f"Área (≈cnt)     = {area:.0f}")

print(f"χ²        = {chi_2:.2f}")
print(f"gl (dof)  = {dof}")
print(f"χ²/dof    = {chi_2_red:.2f}")
print(f"p-value   = {p_value:.3f}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(9, 6), gridspec_kw={"height_ratios": [3, 1]})
xg = np.linspace(x.min(), x.max(), 400)
ax1.errorbar(x, y, yerr=sy, fmt=".", ms=4, capsize=2, label="dados")
ax1.plot(xg, gauss_const(xg, *popt), "r-", label="ajuste")
ax1.axvline(mu, color="r", ls="--", alpha=0.5)
ax1.set_ylabel("contagens")
ax1.legend()
ax1.set_title(f"Cs-137 — μ = {mu:.1f} ± {perr[1]:.1f} canal,  FWHM = {FWHM:.1f}")

res = (y - gauss_const(x, *popt)) / sy
ax2.axhline(0, color="r")
ax2.plot(x, res, "k.")
ax2.set_ylabel("resíduo / σ")
ax2.set_xlabel("canal")
fig.tight_layout()


### Experimenta: o que pode correr mal?

**(a) Esquecer o fundo.** Ajusta o mesmo pico **sem** o termo $c_0$ — vê como a área "rouba" o continuum.

**(b) Estimativa inicial má.** O `curve_fit` precisa de `p0` razoável. Sem ele, ou pode não convergir, ou converge para um mínimo local.

## 3.2 — Lorentziana: o mesmo pico com outro modelo

A lorentziana tem **caudas mais longas** que a gaussiana. Para um fotopico de cintilador a gaussiana é fisicamente melhor (resolução ≡ flutuação estatística do número de fotões), mas é instrutivo comparar.

In [ ]:
def lorentz_const(x, A, x0, gamma, c0):
    return A * gamma**2 / ((x - x0) ** 2 + gamma**2) + c0


p0L = [y.max() - np.median(y), x[y.argmax()], 10, np.median(y)]
poptL, pcovL = curve_fit(lorentz_const, x, y, sigma=sy, p0=p0L, absolute_sigma=True)

chi2_G = np.sum(((y - gauss_const(x, *popt)) / sy) ** 2)
chi2_L = np.sum(((y - lorentz_const(x, *poptL)) / sy) ** 2)
dof = len(x) - 4
print(f"χ²/dof  Gaussiana   = {chi2_G / dof:.2f}")
print(f"χ²/dof  Lorentziana = {chi2_L / dof:.2f}")

fig, ax = plt.subplots()
ax.errorbar(x, y, yerr=sy, fmt=".", ms=4, capsize=2, color="k", alpha=0.5)
ax.plot(xg, gauss_const(xg, *popt), "r-", label=f"Gauss  χ²/dof={chi2_G / dof:.2f}")
ax.plot(xg, lorentz_const(xg, *poptL), "b-", label=f"Lorentz χ²/dof={chi2_L / dof:.2f}")
ax.set_xlabel("canal")
ax.set_ylabel("contagens")
ax.legend()
plt.show()


In [ ]:
def dois_gauss(x, A1, mu1, s1, A2, mu2, s2, c0):
    g1 = A1 * np.exp(-0.5 * ((x - mu1) / s1) ** 2)
    g2 = A2 * np.exp(-0.5 * ((x - mu2) / s2) ** 2)
    return g1 + g2 + c0


mask = (canal > 540) & (canal < 700)
x, y, sy = canal[mask], contagens[mask], sigma_y[mask]

p0 = [80, 584, 13, 70, 663, 13, np.median(y)]
popt, pcov = curve_fit(dois_gauss, x, y, sigma=sy, p0=p0, absolute_sigma=True)
perr = np.sqrt(np.diag(pcov))

for nome, val, err in zip(["A1", "μ1", "σ1", "A2", "μ2", "σ2", "c0"], popt, perr):
    print(f"{nome:>3s} = {val:8.2f} ± {err:.2f}")

xg = np.linspace(x.min(), x.max(), 600)
fig, ax = plt.subplots()
ax.errorbar(x, y, yerr=sy, fmt=".", ms=3, capsize=1, color="k", alpha=0.5, label="dados")
ax.plot(xg, dois_gauss(xg, *popt), "r-", label="ajuste total")
g1 = popt[0] * np.exp(-0.5 * ((xg - popt[1]) / popt[2]) ** 2) + popt[6]
g2 = popt[3] * np.exp(-0.5 * ((xg - popt[4]) / popt[5]) ** 2) + popt[6]
ax.plot(xg, g1, "g--", alpha=0.7, label="Co-60 (1173 keV)")
ax.plot(xg, g2, "b--", alpha=0.7, label="Co-60 (1332 keV)")
ax.set_xlabel("canal")
ax.set_ylabel("contagens")
ax.legend()

In [ ]:
plt.errorbar(canal, contagens, yerr=sigma_y, fmt=".", ms=2, lw=0.5, ecolor="gray", label=r"dados $\pm~\sqrt{N}$")
plt.xlabel("Canal")
plt.ylabel("Contagens")
plt.title(r"Espectro do MCA ($^{137}$Cs + $^{60}$Co)")
plt.axvline(328, color="r", ls="--", label="Cs-137 (662 keV)")
plt.axvline(584, color="g", ls="--", label="Co-60 (1173 keV)")
plt.axvline(663, color="b", ls="--", label="Co-60 (1332 keV)")
plt.legend()
plt.show()

# Exercício Final: Fazer o espectro energético.
Sabendo que os picos no espectro de contagens correspondem às energias:
- Cs-137: 662 keV
- Co-60: 1173 keV e 1332 keV

1) Obtem o canal correspondente a cada pico, assim como a incerteza associada. 
2) Utiliza os canais e as energias para fazer um gráfico de energia vs canal, e ajusta uma função linear para obter a calibração do MCA (energia por canal). Qual é a energia correspondente ao canal 0? E ao canal 1024?
3) Com a calibração, converte o espectro de contagens por canal para um espectro de contagens por energia. Qual é a energia correspondente ao pico do Cs-137? E aos picos do Co-60? Estão onde deveriam estar? 

Não te esqueças de incluir as barras de erro! E de mostrar o gráfico do espectro energético, com os picos identificados.